## We seek to first apporximate the current Pixel Threshold by identifying the partition of greatest length within the current DES DR2 Catalog 

In [12]:
import sys
import os

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from pathlib import Path
import importlib

import numpy as np
import pandas as pd
from astropy.io import ascii
import matplotlib.pyplot as plt

from dask.distributed import Client
import dask.array
from dask.dataframe.utils import make_meta

from hats import read_hats
from hats.inspection import plot_pixels
from hats_import.catalog.file_readers import CsvReader
from hats_import.margin_cache.margin_cache_arguments import MarginCacheArguments
from hats_import.pipeline import ImportArguments, pipeline_with_client

import lsdb

from catalog_filtering import bandFilterLenient, contains_PM
import hpms_pipeline as hpms

print("Imported libraries.")

Imported libraries.


In [14]:
from hats_import.pipeline import pipeline_with_client
from hats_import.catalog.arguments import ImportArguments

In [15]:
CATALOG_DIR = Path("../../../catalogs")
MARGIN_CACHE_DIR = CATALOG_DIR / 'margin_caches'

DES_NAME = "des_light"
DES_DIR = CATALOG_DIR / DES_NAME 

DES_MARGIN_CACHE_NAME = "des_margin_cache_18_arcsec"
DES_MARGIN_CACHE_DIR = MARGIN_CACHE_DIR / DES_MARGIN_CACHE_NAME

In [16]:
des_dr2 = lsdb.read_hats(DES_DIR)
des_dr2

,CLASS_STAR_G,CLASS_STAR_R,CLASS_STAR_I,CLASS_STAR_Z,CLASS_STAR_Y,FLAGS_G,FLAGS_R,FLAGS_I,FLAGS_Z,FLAGS_Y,RA,DEC,COADD_OBJECT_ID,SPREAD_MODEL_G,SPREAD_MODEL_R,SPREAD_MODEL_I,SPREAD_MODEL_Z,SPREAD_MODEL_Y,WAVG_MAG_PSF_G,WAVG_MAG_PSF_R,WAVG_MAG_PSF_I,WAVG_MAG_PSF_Z,WAVG_MAG_PSF_Y,WAVG_MAGERR_PSF_G,WAVG_MAGERR_PSF_R,WAVG_MAGERR_PSF_I,WAVG_MAGERR_PSF_Z,WAVG_MAGERR_PSF_Y,NEPOCHS_G,NEPOCHS_R,NEPOCHS_I,NEPOCHS_Z,NEPOCHS_Y
npartitions=1582,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 4, Pixel: 0",double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow]
"Order: 5, Pixel: 8",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 743",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 1, Pixel: 47",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [17]:
with Client():
    partition_lengths = des_dr2.map_partitions(len).compute()
partition_lengths 

/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/dask/dataframe/dask_expr/_expr.py:4078: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/lsdb/catalog/dataset/healpix_dataset.py:534: RuntimeWarning: output of the function must be a DataFrame to generate an LSDB `Catalog`. `map_partitions` will return a dask object instead of a Catalog.
  warnings.warn(


0    838930
0    408866
      ...  
0      2926
0    289866
Length: 1582, dtype: int64

In [18]:
partition_lengths.max()

np.int64(971062)

## Let us assume that the current Pixel Threshold is $\bf{1,000,000}$, let's change this to $\bf{100,000}$

In [27]:
# Reimport Args:

output_name = "des_dr2_pix_thresh_100k"
OUTPUT_DIR = CATALOG_DIR
output_artifact_name= output_name
pixel_thresh=100000
highest_healpix_order = 10

args = ImportArguments.reimport_from_hats(
        DES_DIR,
        OUTPUT_DIR,
        output_artifact_name=output_name,
        pixel_threshold=pixel_thresh,
        highest_healpix_order=10
)


Validating catalog at path ../../../catalogs/des_light ... 
Found 1582 partitions.
Approximate coverage is 30.99 % of the sky.


In [28]:
with Client(n_workers = 10) as client:
    display(client)
    pipeline_with_client(args, client)

/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33565 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:33565/status,
Dashboard: http://127.0.0.1:33565/status,Workers: 10
Total threads: 130,Total memory: 234.38 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45711,Workers: 10
Dashboard: http://127.0.0.1:33565/status,Total threads: 130
Started: Just now,Total memory: 234.38 GiB
Comm: tcp://127.0.0.1:37697,Total threads: 13
Dashboard: http://127.0.0.1:35591/status,Memory: 23.44 GiB
Nanny: tcp://127.0.0.1:38215,


Planning  :   0%|          | 0/4 [00:00<?, ?it/s]

Mapping   :   0%|          | 0/1582 [00:00<?, ?it/s]

Binning   :   0%|          | 0/2 [00:00<?, ?it/s]

Splitting :   0%|          | 0/1582 [00:00<?, ?it/s]

Reducing  :   0%|          | 0/23044 [00:00<?, ?it/s]

Finishing :   0%|          | 0/5 [00:00<?, ?it/s]

2025-10-04 22:16:02,847 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,847 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,848 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,849 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,849 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,850 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,850 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,851 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04 22:16:02,852 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2025-10-04

In [30]:
des_pix_100k = lsdb.read_hats(OUTPUT_DIR / output_name)
des_pix_100k

,CLASS_STAR_G,CLASS_STAR_R,CLASS_STAR_I,CLASS_STAR_Z,CLASS_STAR_Y,FLAGS_G,FLAGS_R,FLAGS_I,FLAGS_Z,FLAGS_Y,RA,DEC,COADD_OBJECT_ID,SPREAD_MODEL_G,SPREAD_MODEL_R,SPREAD_MODEL_I,SPREAD_MODEL_Z,SPREAD_MODEL_Y,WAVG_MAG_PSF_G,WAVG_MAG_PSF_R,WAVG_MAG_PSF_I,WAVG_MAG_PSF_Z,WAVG_MAG_PSF_Y,WAVG_MAGERR_PSF_G,WAVG_MAGERR_PSF_R,WAVG_MAGERR_PSF_I,WAVG_MAGERR_PSF_Z,WAVG_MAGERR_PSF_Y,NEPOCHS_G,NEPOCHS_R,NEPOCHS_I,NEPOCHS_Z,NEPOCHS_Y
npartitions=23044,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
"Order: 6, Pixel: 0",double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],int16[pyarrow],double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],double[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow],int32[pyarrow]
"Order: 7, Pixel: 6",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 6, Pixel: 48136",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 3, Pixel: 767",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [31]:
with Client(n_workers=8) as client:
    display(client)
    partition_lengths = des_pix_100k.map_partitions(len).compute()

/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44223 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:44223/status,
Dashboard: http://127.0.0.1:44223/status,Workers: 8
Total threads: 128,Total memory: 234.38 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36339,Workers: 8
Dashboard: http://127.0.0.1:44223/status,Total threads: 128
Started: Just now,Total memory: 234.38 GiB
Comm: tcp://127.0.0.1:38597,Total threads: 16
Dashboard: http://127.0.0.1:46323/status,Memory: 29.30 GiB
Nanny: tcp://127.0.0.1:34539,


/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/dask/dataframe/dask_expr/_expr.py:4078: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/lsdb/catalog/dataset/healpix_dataset.py:534: RuntimeWarning: output of the function must be a DataFrame to generate an LSDB `Catalog`. `map_partitions` will return a dask object instead of a Catalog.
  warnings.warn(
/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/client.py:3383: UserWarning: Sending large graph of size 10.74 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data 

TimeoutError: 